# Step 2: Recommendation Engine
In this notebook, we will create a **Recommendation Engine** to map each predicted disease to relevant:
- Descriptions
- Medications
- Diet plans
- Precautions
- Workouts

These mappings will be sourced from the CSV files provided in the dataset folder, and the final output will be a function that returns all related recommendations for a disease as a dictionary.

In [1]:
# Import necessary libraries
import pandas as pd
import ast  # For converting string list to actual Python list

## Load Supporting CSV Files
Each of these CSVs maps diseases to some kind of advice or recommendation. We will now load all of them and inspect their contents.

In [2]:
# Load description file
desc_df = pd.read_csv('/content/description.csv')
desc_df.head()

,Disease,Description
0,Fungal infection,Fungal infection is a common skin condition ca...
1,Allergy,Allergy is an immune system reaction to a subs...
2,GERD,GERD (Gastroesophageal Reflux Disease) is a di...
3,Chronic cholestasis,Chronic cholestasis is a condition where bile ...
4,Drug Reaction,Drug Reaction occurs when the body reacts adve...


In [3]:
# Load medications file
med_df = pd.read_csv('/content/medications.csv')
med_df.head()

,Disease,Medication
0,Fungal infection,"['Antifungal Cream', 'Fluconazole', 'Terbinafi..."
1,Allergy,"['Antihistamines', 'Decongestants', 'Epinephri..."
2,GERD,"['Proton Pump Inhibitors (PPIs)', 'H2 Blockers..."
3,Chronic cholestasis,"['Ursodeoxycholic acid', 'Cholestyramine', 'Me..."
4,Drug Reaction,"['Antihistamines', 'Epinephrine', 'Corticoster..."


In [4]:
# Load diets file
diet_df = pd.read_csv('/content/diets.csv')
diet_df.head()

,Disease,Diet
0,Fungal infection,"['Antifungal Diet', 'Probiotics', 'Garlic', 'C..."
1,Allergy,"['Elimination Diet', 'Omega-3-rich foods', 'Vi..."
2,GERD,"['Low-Acid Diet', 'Fiber-rich foods', 'Ginger'..."
3,Chronic cholestasis,"['Low-Fat Diet', 'High-Fiber Diet', 'Lean prot..."
4,Drug Reaction,"['Antihistamine Diet', 'Omega-3-rich foods', '..."


In [5]:
# Load precautions file
prec_df = pd.read_csv('/content/precautions_df.csv')
prec_df.head()

,Unnamed: 0,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [6]:
# Load workout file
work_df = pd.read_csv('/content/workout_df.csv')
work_df.head()

,Unnamed: 0.1,Unnamed: 0,disease,workout
0,0,0,Fungal infection,Avoid sugary foods
1,1,1,Fungal infection,Consume probiotics
2,2,2,Fungal infection,Increase intake of garlic
3,3,3,Fungal infection,Include yogurt in diet
4,4,4,Fungal infection,Limit processed foods


## Build the Unified Recommendation Function
This function will take a disease name (predicted by the model) and fetch its related information from all the loaded dataframes.

In [7]:
def get_all_recommendations(disease_name):
    """
    Fetches all recommendations (description, medications, diet, precautions, workouts) for a given disease.
    """
    result = {}

    # Description
    try:
        result['description'] = desc_df.loc[desc_df['Disease'] == disease_name, 'Description'].values[0]
    except IndexError:
        result['description'] = 'No description available.'

    # Medications (convert string to list if needed)
    try:
        meds = med_df.loc[med_df['Disease'] == disease_name, 'Medication'].values[0]
        result['medications'] = ast.literal_eval(meds)
    except (IndexError, ValueError):
        result['medications'] = []

    # Diet
    try:
        diet = diet_df.loc[diet_df['Disease'] == disease_name, 'Diet'].values[0]
        result['diet'] = ast.literal_eval(diet)
    except (IndexError, ValueError):
        result['diet'] = []

    # Precautions (4 columns: Precaution_1 to Precaution_4)
    try:
        prec_row = prec_df.loc[prec_df['Disease'] == disease_name].iloc[:, 2:].values.flatten()
        result['precautions'] = [p for p in prec_row if str(p) != 'nan']
    except IndexError:
        result['precautions'] = []

    # Workouts (may have multiple entries per disease)
    try:
        workouts = work_df.loc[work_df['disease'] == disease_name, 'workout'].tolist()
        result['workouts'] = workouts
    except:
        result['workouts'] = []

    return result

## Test the Recommendation Function
We will test our function with an example disease: `Fungal infection`.

In [8]:
# Test with an example
sample_disease = 'Fungal infection'
recommendations = get_all_recommendations(sample_disease)
recommendations

{'description': 'Fungal infection is a common skin condition caused by fungi.',
 'medications': ['Antifungal Cream',
  'Fluconazole',
  'Terbinafine',
  'Clotrimazole',
  'Ketoconazole'],
 'diet': ['Antifungal Diet',
  'Probiotics',
  'Garlic',
  'Coconut oil',
  'Turmeric'],
 'precautions': ['bath twice',
  'use detol or neem in bathing water',
  'keep infected area dry',
  'use clean cloths'],
 'workouts': ['Avoid sugary foods',
  'Consume probiotics',
  'Increase intake of garlic',
  'Include yogurt in diet',
  'Limit processed foods',
  'Stay hydrated',
  'Consume green tea',
  'Eat foods rich in zinc',
  'Include turmeric in diet',
  'Eat fruits and vegetables']}

## ✅ Summary
- We loaded all auxiliary medical recommendation files.
- Built a function `get_all_recommendations()` to fetch and return all data.
- Successfully tested it with a sample disease.

**Next Step**: We'll integrate this logic into our Flask web app in the next notebook: `3_flask_app.ipynb`.
That app will:
- Take symptom input
- Use the trained model to predict a disease
- Display all these recommendations in a user-friendly web interface.